In [1]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 447, done.
remote: Counting objects: 100% (447/447), done.
remote: Compressing objects: 100% (342/342), done.
remote: Total 447 (delta 105), reused 335 (delta 87), pack-reused 0 (from 0)
Receiving objects: 100% (447/447), 5.15 MiB | 5.46 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [9]:
%cd LLaMA-Factory



/Users/artemdzalilov/working/VSEROSII_first/baselines/llm_training_and_inference/LLaMA-Factory


In [3]:
!pip install -e ".[torch,metrics]" --no-build-isolation
!pip install liger-kernel

Obtaining file:///Users/artemdzalilov/working/VSEROSII_first/baselines/llm_training_and_inference/LLaMA-Factory
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.4.dev0-0.editable-py3-none-any.whl size=28939 sha256=0b7e8a683ad66f16f367ee235b8c7d8656ba622907a30ccf68d1b76a78b791d2
  Stored in directory: /private/var/folders/0k/z_l4ql_113943rc3bncsv20r0000gn/T/pip-ephem-wheel-cache-4gbj09q5/wheels/d7/a6/2e/d5ff503bfe18119aabd940ef8abe538f5ac91fb85f7d34f206
Successfully built llamafactory
  Attempting uninstall: llamafactory
    Found existing installation: llamafactory 0.9.4.dev0
    Uninstalling llamafactory-0.9.4.dev0:
      Successfully uninstalled llamafactory-0.9.4.dev0

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
  Using cached lig

In [4]:
!pip install -U ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [5]:
!python -V

Python 3.11.13


In [10]:
import json
from PIL import Image
import pandas as pd
import os
import io
import uuid
from PIL import Image
import numpy as np


In [11]:

def save_image(img_input, save_dir: str, ext: str = "png") -> str:
    os.makedirs(save_dir, exist_ok=True)

    filename = f"{uuid.uuid4().hex}.{ext}"
    save_path = os.path.join(save_dir, filename)

    if isinstance(img_input, Image.Image):
        img = img_input

    elif isinstance(img_input, bytes):
        img = Image.open(io.BytesIO(img_input))

    elif isinstance(img_input, np.ndarray):
        if img_input.ndim == 3 and img_input.shape[2] == 3:
            img = Image.fromarray(img_input[..., ::-1])
        else:
            img = Image.fromarray(img_input)

    elif isinstance(img_input, str) and os.path.exists(img_input):
        img = Image.open(img_input)

    else:
        raise TypeError(f"Неподдерживаемый тип входных данных: {type(img_input)}")

    img.save(save_path)
    return os.path.abspath(save_path)

In [12]:
INSTRUCTION_COLUMN = 'instruction'
INPUT_COLUMN = 'input'
OUTPUT_COLUMN = 'output'
IMAGE_COLUMN = 'image'
MODEL_DICT = {'model_name': 'Qwen/Qwen3-VL-2B-Instruct', 'model_template': 'qwen3_vl_nothink'}

In [13]:
df = pd.read_csv('/Users/artemdzalilov/working/VSEROSII_first/baselines/llm_training_and_inference/train.csv')
data = df.to_dict('records')

In [14]:
df.columns

Index(['instruction', 'input', 'output', 'image'], dtype='object')

In [15]:
def create_llama_factory_dataset_from_json(data, instruction_column, message_column, output_column, image_column=None):
    res = []
    for i in data:
        try:
            train_sample = {
                'instruction': i[instruction_column],
                'input': i[message_column],
                'output': i[output_column],
            }
            if image_column and i.get(image_column): 
                train_sample['images'] = [i[image_column]]
                train_sample['input'] = '<image>' + train_sample['input']
            res.append(train_sample)
        except Exception as e:
            print(f'Error while processing {i}: {e}')
    return res

dataset = create_llama_factory_dataset_from_json(data, INSTRUCTION_COLUMN, INPUT_COLUMN, OUTPUT_COLUMN, IMAGE_COLUMN)

with open('./data/my_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

In [16]:
with open('./data/dataset_info.json', 'r') as f:
    data_info = json.load(f)

In [17]:
if IMAGE_COLUMN:
  data_info['my_dataset'] = {
      "file_name": "my_dataset.json",
      "columns":{
          "system":"instruction",
          "prompt":"input",
          "response":"output",
          "images":"images"
          }
  }
else:
  data_info['my_dataset'] = {
      "file_name": "my_dataset.json",
      "columns":{
          "system":"instruction",
          "prompt":"input",
          "response":"output",
          }
    }
with open('./data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(data_info, f, indent=4, ensure_ascii=False)

In [18]:
args = dict(
  stage="sft",                      
  do_train=True,
  model_name_or_path=MODEL_DICT['model_name'], 
  dataset="my_dataset",            
  template=MODEL_DICT['model_template'],               
  finetuning_type="lora",                 
  lora_target="all", 
  output_dir="trained_model_lora",            
  per_device_train_batch_size=1, 
  gradient_accumulation_steps=2,          
  lr_scheduler_type="cosine",         
  logging_steps=5,      
  warmup_ratio=0.1,              
  save_strategy='epoch',
  save_steps=0,                  
  learning_rate=5e-5,          
  num_train_epochs=1.0,                    
  max_grad_norm=1.0,              
  loraplus_lr_ratio=16.0,                 
  #fp16=True,                 
  #enable_liger_kernel=True,
  # enable_thinking=True
  cutoff_len=4096
)

if IMAGE_COLUMN:
    args["image_max_pixels"]=334*334

In [19]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ['OMP_NUM_THREADS'] = "1"

In [20]:
json.dump(args, open("model_training.json", "w", encoding="utf-8"), indent=2)

In [25]:
!llamafactory-cli train model_training.json

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
[WARNING|2025-11-16 12:20:41] llamafactory.hparams.parser:148 >> We recommend enable mixed precision training.
[INFO|2025-11-16 12:20:41] llamafactory.hparams.parser:468 >> Process rank: 0, world size: 1, device: mps, distributed training: False, compute dtype: None
[INFO|tokenization_utils_base.py:2095] 2025-11-16 12:20:43,010 >> loading file vocab.json from cache at /Users/artemdzalilov/.cache/huggingface/hub/models--Qwen--Qwen3-VL-2B-Instruct/snapshots/89644892e4d85e24eaac8bacfd4f463576704203/vocab.json
[INFO|tokenization_utils_base.py:2095] 2025-11-16 12:20:43,010 >> loading file merges.txt from cache at /Users/artemdzalilov/.cache/huggingface/hub/models--Qwen--Qwen3-VL-2B-Instruct/snapshots/89644892e4d85e24eaac8bacfd4f463576704203/merges.txt
[INFO|tokenization_utils_base.py:2095] 2025-

In [33]:
args = dict(
    model_name_or_path = MODEL_DICT['model_name'], 
    adapter_name_or_path = "trained_model_lora", 
    template = MODEL_DICT['model_template'],
    finetuning_type = "lora",
    export_dir = "trained_model_lora_merged",
    export_size = 2, 
    export_device ="cpu", 
)

with open("merge_model.json", "w", encoding="utf-8") as f: 
    json.dump(args, f, ensure_ascii=False, indent=4)

In [34]:
!llamafactory-cli export merge_model.json

/Users/artemdzalilov/working/VSEROSII_first/venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[INFO|tokenization_utils_base.py:2095] 2025-11-13 01:34:05,754 >> loading file vocab.json from cache at /Users/artemdzalilov/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/vocab.json
[INFO|tokenization_utils_base.py:2095] 2025-11-13 01:34:05,754 >> loading file merges.txt from cache at /Users/artemdzalilov/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca/merges.txt
[INFO|tokenization_utils_base.py:2095] 2025-11-13 01:34:05,754 >> loading file tokenizer.json from cache at /Users/artemdzalilov/.cache/huggingface/hub/models--Qwen--

In [ ]:
1/0

#Quantize model

In [35]:
!pip install llmcompressor

  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
Using cached loguru-0.7.3-py3-none-any.whl (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 4.4 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 1.5 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 1.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: torch90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/8 [numpy]
    Found existing installation: torch 2.9.0━━━━━━━━━━━━━━━━━━ 1/8 [numpy]
    Uninstalling torch-2.9.0:90m╺━━━━━━━━━━━━━━━━━━━━━━━━ 3/8 [torch]
      Successfully uninstalled torch-2.9.0━━━━━━━━━━━━━━━━━━━━ 3/8 [torch]
  Attempting uninstall: accelerate━━━━━━━━━━━━━━━━━━━━━━━━ 3/8 [torch]
    Found existing installation: accelerate 1.11.0━━━━━━━━━━━━ 3/8 [torch]
    Uninstalling accelerate-1.

In [1]:
!pip install --no-cache-dir transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.9 kB/s  0:01:14m0:00:03:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 48.7 kB/s  0:00:10 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 92.2 kB/s  0:00:37m0:00:0100:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llamafactory 0.9.4.dev0 requires sentencepiece, which is not installed.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.10.0 which is incompatible.
trl 0.9.6 requires numpy<2.0.0,>=1.18.2, but you have numpy 2.3.4 which is incompatible.
llmcompressor 0.8.1 requires numpy<=2.3.3,>=2.0.0, but you have numpy 2.3.4 which is incompatible.
llmcompressor 0.8.1 requires transformers<=4.56.2,>=4.53.0, but you have transformers 4.57.1 which is incomp

In [1]:
import os
import torch
from datasets import load_dataset
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor import oneshot
from PIL import Image
import json
from llmcompressor.modifiers.awq import AWQModifier

/Users/artemdzalilov/working/VSEROSII_first/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: Could not import module 'AutoProcessor'. Are this object's requirements defined correctly?

In [2]:
import transformers
print(transformers.__file__)
print(transformers.__version__)

/Users/artemdzalilov/working/VSEROSII_first/venv/lib/python3.11/site-packages/transformers/__init__.py
4.57.1


In [ ]:
CONFIG = {
    "model_name": "./trained_model_lora_merged",
    "dataset_paths": [
        "./data/my_dataset.json",
    ],
    "output_dir": "./model_quantizes",
    "num_calibration_samples": 128,
    "max_seq_length": 4096,
    "batch_size": 1, #Important
}

In [ ]:
def load_and_merge_datasets(dataset_paths: list):
    if not isinstance(dataset_paths, list):
        raise TypeError("dataset_paths должен быть списком")

    print(f"📂 Загрузка и объединение датасетов ({len(dataset_paths)} файлов)...")
    combined_dataset = load_dataset("json", data_files=dataset_paths, split="train")
    print(f"✅ Загружено {len(combined_dataset)} примеров")
    return combined_dataset

In [ ]:
def preprocess_multimodal_data(examples, images_base_paths):
    processed_texts, processed_images = [], []

    for instruction, input_text, image_rel_path in zip(
        examples.get("instruction", []),
        examples.get("input", []),
        examples.get("image", [None] * len(examples.get("instruction", []))),
    ):
        full_text = f"{instruction}\n\n{input_text}"
        processed_texts.append(full_text)

        image_found = None
        if image_rel_path:
            for base_dir in images_base_paths:
                candidate = os.path.join(base_dir, image_rel_path)
                if os.path.exists(candidate):
                    try:
                        image_found = Image.open(candidate).convert("RGB")
                    except Exception as e:
                        print(f"⚠️ Ошибка при открытии {candidate}: {e}")
                    break
        processed_images.append(image_found)

    return {"text": processed_texts, "images": processed_images}

In [ ]:
def create_calibration_dataset(dataset, images_base_paths, num_samples):
    print(f"🔧 Подготовка {num_samples} примеров для калибровки...")
    if len(dataset) > num_samples:
        dataset = dataset.shuffle(seed=42).select(range(num_samples))

    def _map(examples):
        return preprocess_multimodal_data(examples, images_base_paths)

    calibration_dataset = dataset.map(
        _map,
        batched=True,
        batch_size=4,
        remove_columns=dataset.column_names,
        desc="Предобработка мультимодальных данных"
    )

    print(f"✅ Датасет готов: {len(calibration_dataset)} примеров.")
    return calibration_dataset


def create_smoothquant_recipe():
    recipe = AWQModifier(
        targets="Linear",
        scheme="W8A16",
        ignore=["re:.*lm_head", "re:.visual.", "re:.*mlp.gate$"],
        duo_scaling=False,
    )
    return recipe

In [ ]:
def save_quantization_info(output_dir, config):
    info = {
        "model": config["model_name"],
        "quantization_method": "SmoothQuant W8A8 (мультимодальная)",
        "num_calibration_samples": config["num_calibration_samples"],
        "max_seq_length": config["max_seq_length"],
        "smoothquant_alpha": config["smoothquant_alpha"],
        "datasets": config["dataset_paths"],
    }

    path = os.path.join(output_dir, "quantization_info.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)
    print(f"📋 Информация сохранена: {path}")

In [ ]:
def quantize_multimodal_model(
    model_name,
    dataset,
    images_base_paths,
    output_dir,
    num_calibration_samples,
    max_seq_length,
    smoothquant_alpha,
):
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📦 Загрузка модели и процессора: {model_name}")
    processor = AutoProcessor.from_pretrained(model_name)
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_name, torch_dtype=torch.float16, trust_remote_code=True
    )

    print(f"\n🧪 Подготовка мультимодальных примеров ({num_calibration_samples})...")
    if len(dataset) > num_calibration_samples:
        dataset = dataset.shuffle(seed=42).select(range(num_calibration_samples))

    def _map_fn(examples):
        processed = preprocess_multimodal_data(examples, images_base_paths)
        flg = 0

        for i in processed["images"]:
            if i is None:
                flg = 1
        print(f'FLAG: {flg}')
        if flg==0:
            inputs = processor(
                text=processed["text"],
                images=processed["images"],
                padding=True,
                max_length=max_seq_length,
                return_tensors=None,
            )
        else:
            inputs = processor(
                text=processed["text"],
                images=None,
                padding=True,
                max_length=max_seq_length,
                return_tensors=None, 
            )
        return inputs

    calibration_dataset = dataset.map(
        _map_fn,
        batched=True,
        batch_size=2,
        desc="📋 Формирование тензорного датасета для калибровки",
        remove_columns=dataset.column_names,
    )

    print(f"✅ Датасет готов: {len(calibration_dataset)} примеров.")

    recipe = create_smoothquant_recipe(alpha=smoothquant_alpha)

    print("\n⚙️ Запуск квантизации (oneshot)...")

    oneshot(
        model=model,
        dataset=calibration_dataset,
        num_calibration_samples=num_calibration_samples,
        recipe=recipe,
        output_dir=output_dir,
        max_seq_length=max_seq_length,
        pipeline="sequential",
    )

    print("\n✅ Квантизация завершена успешно!")
    print(f"📁 Модель сохранена в: {output_dir}")

    save_quantization_info(output_dir, CONFIG)


In [ ]:
def main():
    print("=" * 80)
    print("🎯 КВАНТИЗАЦИЯ Qwen3-VL С ИЗОБРАЖЕНИЯМИ")
    print("=" * 80)

    dataset = load_and_merge_datasets(CONFIG["dataset_paths"])

    quantize_multimodal_model(
        model_name=CONFIG["model_name"],
        dataset=dataset,
        images_base_paths=CONFIG["images_base_paths"],
        output_dir=CONFIG["output_dir"],
        num_calibration_samples=CONFIG["num_calibration_samples"],
        max_seq_length=CONFIG["max_seq_length"],
        smoothquant_alpha=CONFIG["smoothquant_alpha"],
    )

    print("\n🎉 ВСЁ ГОТОВО!")


if __name__ == "__main__":
    print(f"🔧 CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
        print(f"💾 Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    main()